# 第九部分：Agent 的诞生 —— 解码、微调与结构化约束

> **本章目标**：将预训练模型改造为听你话、不乱说、会调API的生产级Agent。

## 第33章　解码策略的"人格分裂"实验 —— 从贪心到核采样

> 本章目标：不只讲采样公式，而是带着读者在"输出质量"和"多样性"之间做工程调参实验。亲手操控 Temperature、Top-k、Top-p、Repetition Penalty 四个旋钮，观察同一个 Prompt 在四种策略下生成的文本品质差异。
> 前置知识：第 15 章（采样策略）、第 22 章（softmax/Logit Bias）、第 32 章（KV Cache 推理）

> 🕵️ **开篇导语：审讯策略的"人格分裂"实验**
>
> 设备推理飞快，但它面临一个更微妙的问题：**每次只输出一个 Token，该选哪个？**
>
> 如果永远选概率最高的那个（**Greedy 策略**），你会发现它像一位固执的老刑警，翻来覆去只说同一句话——"我怀疑是你，就是你，肯定是你"。生成 50 个 Token 里，30 个是重复的 n-gram。它掉进了"复读机诅咒"。
>
> 如果反过来，完全随机选（**高 Temperature**），它又像一位思维过于发散的实习生——上一句还在分析指纹，下一句突然讨论晚餐吃什么。概率分布被 Temperature 抹平后，所有 Token 的机会均等，输出的文本完全失控。
>
> **Top-k** 试图在两者之间找平衡：只从概率最高的前 k 个里选。但 `k=50` 是暴力截断——如果前 50 个只覆盖了 60% 的概率质量，剩余的 40% 被硬生生抛弃（截断损失）。
>
> **Top-p（核采样）** 是更优雅的方案：从概率最高的 Token 开始累加，直到累计概率达到 `p=0.9`。候选集随上下文动态变化——信息量大时候选集宽，信息量小时候选集窄。这就是当前所有 SOTA Agent 的默认"性格开关"。
>
> 而为了防止 Agent 在长思考（CoT）中车轱辘话来回说，我们还要在 logits 层装一道**惩罚机制（Repetition Penalty）**：对已经说过的 Token 做减法，强迫它换话题。
>
> 本章没有"正确答案"——只有"合适的选择"。你将亲手操控这些旋钮，观察同一个 Prompt 在四种策略下分裂出四种截然不同的"人格"。而你将学会的，是如何为你的 Agent 选择最匹配的那一个。



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("env ready")


### 33.1　Greedy 的"复读机"诅咒

Greedy 策略永远选概率最高的 Token——`argmax(logits)`。每次推理唯一确定，输出完全可复现。这对工具调用（Function Calling）是优点——同样的输入给同样的动作。但对创意生成是灾难——模型陷入"最安全"的循环。

现象：用 GPT-2 做 Greedy decoding，"我不 知道 我不 知道 我不 知道"——模型反复输出最高概率的 token 组合，因为一旦一个模式开始重复，它就自我强化。

📐 **定义　Greedy Decoding**：`token_t = argmax(softmax(logits_t))`。确定性输出，无随机性。Distinct-n 指标（连续 n 个 token 中不重复的比例）最低。

💻 **代码　Greedy 生成 50 个 Token，观察重复模式**


In [ ]:
import numpy as np

np.random.seed(42)
vocab_size, seq_len = 100, 50

# 模拟一个偏斜的概率分布——最高概率 token 始终是 0
logits = np.random.randn(seq_len, vocab_size) * 0.5
logits[:, 0] += 2.0  # 让 token 0 始终是最高概率

# Greedy: 永远选 argmax
tokens = np.argmax(logits, axis=1)

# 统计重复
repeats = sum(1 for i in range(1, len(tokens)) if tokens[i] == tokens[i-1])
unique_ratio = len(np.unique(tokens)) / len(tokens)

print(f"Greedy 生成的 token 序列: {tokens}")
print(f"唯一 token 比例: {unique_ratio:.0%}")
print(f"相邻重复次数: {repeats}/{seq_len-1}")
print(f"→ Greedy 陷入重复循环——复读机诅咒")


> **关键洞察**：Greedy 的致命缺陷在于**没有探索**——一旦某个 token 的 logit 略微领先，它就永远被选中。在长文本生成中，这导致 n-gram 严重重复。但 Greedy 也有用武之地：工具调用（第 35 章）中需要确定性输出，T=0 或 Greedy 是最佳选择。

---


### 33.2　Temperature 的混沌效应

Temperature 控制 softmax 的"锐度"：`softmax(logits / T)`。

- **T → 0**：退化为 Greedy——概率全部集中在最大值
- **T = 1**：标准 softmax——原始概率分布
- **T → ∞**：趋近均匀分布——所有 token 等概率

T < 1 让高概率 token 更突出（保守、确定），T > 1 让概率更平滑（多样、有创造力）。

📐 **定义　Temperature**：`p_i = exp(z_i/T) / Σ exp(z_j/T)`。T 控制 softmax 输出的熵——T 越大熵越大（越随机），T 越小熵越小（越确定）。实践中 T=0.7~0.9 是创意写作的常见甜点区间。

💻 **代码　Temperature 对概率分布的影响（四子图对比）**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
logits = np.array([3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.2, 0.1])

def softmax(x, T=1.0):
    x = np.array(x, dtype=np.float64) / T
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

temperatures = [0.1, 0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, T in zip(axes, temperatures):
    probs = softmax(logits, T)
    ax.bar(range(len(logits)), probs, color='steelblue', edgecolor='white')
    entropy = -np.sum(probs * np.log(probs + 1e-8))
    ax.set_title(f'T = {T}\nentropy = {entropy:.2f}')
    ax.set_xlabel('Token rank'); ax.set_ylabel('Probability')

plt.suptitle('Temperature: Control the Entropy of Softmax', fontsize=13)
plt.tight_layout(); plt.show()

print("T=0.1: 几乎确定性 (一个 token 独占概率)")
print("T=0.5: 适度集中 (高概率 token 仍占主导)")
print("T=1.0: 原始分布")
print("T=2.0: 几乎均匀 (所有 token 都有机会)")
print("\n实践: T=0.1~0.3 工具调用 | T=0.7~0.9 创意写作 | T=1.5+ 探索/反思")


> **关键洞察**：Temperature 是整个解码策略的"基础层"——它控制概率分布的基础形状。Top-k 和 Top-p 在 Temperature 之后应用（都是对 softmax 输出的后处理）。所以 Temperature 是"调音"，Top-k/Top-p 是"选音"。

🔗 **AI 连接**：第 35 章 Agent 工具调用时，推荐 T=0.1~0.3 确保 JSON 格式稳定；第 15.6 节的 Agent 视角框给出了三种场景的温度推荐。

---


### 33.3　Top-k 的"截断暴力"

Top-k 只从概率最高的 k 个 token 中采样，其余概率置 0 后重归一化。k=50 是常见默认值。

**问题**：k 是固定的。如果分布很集中（只有 3 个合理选项），k=50 太宽松（引入噪音——低概率的垃圾 token 也有机会被选中）。如果分布很分散（200 个合理选项），k=50 又太严格（截断概率质量——40% 的合理选项被硬生生抛弃）。

这就是"截断损失（Truncation Loss）"：固定 k 导致的概率质量丢失。

📐 **定义　Top-k**：保留概率最高的 k 个 token，其余置零，重归一化。k 过大 → 引入噪音；k 过小 → 截断损失。k=50 是经验默认值，但无法自适应不同分布。

💻 **代码　Top-k 在不同分布上的覆盖率对比**


In [ ]:
import numpy as np

def softmax(x):
    x = np.float64(x); x = x-x.max(); e = np.exp(x); return e/e.sum()

# 集中分布 vs 分散分布
logits_c = np.array([5.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])
logits_f = np.array([2.0, 1.9, 1.8, 1.7, 1.6, 1.5, 1.4, 1.3])

for name, logits in [("集中分布", logits_c), ("分散分布", logits_f)]:
    probs = softmax(logits)
    for k in [3, 5, 10]:
        topk = probs.copy()
        topk[np.argsort(topk)[:-k]] = 0
        coverage = topk.sum()
        print(f"{name}: Top-{k} 覆盖 {coverage:.0%} 的概率质量")
    print()


---


### 33.4　Top-p（核采样）的动态哲学

Top-p（Nucleus Sampling）不固定 k——从概率最高的 token 累加，直到累积概率 ≥ p（如 p=0.9）。候选集大小随上下文**动态变化**。

集中分布时 p=0.9 只需 2 个 token，分散分布时需要 6 个——完美自适应。**p=0.9 是当前所有 SOTA LLM 的默认解码策略。**

📐 **定义　Top-p / 核采样**：从 argmax 开始累加概率，直到 Σp ≥ p_threshold。候选集动态大小，不存在截断损失。p=0.9 为工业标准甜点——既避免 Greedy 的重复，又避免 Top-k 的硬截断。

💻 **代码　Top-p 实现 + 与 Top-k 对比**


In [ ]:
import numpy as np

def softmax(x):
    x = np.float64(x); x = x-x.max(); e = np.exp(x); return e/e.sum()

def top_p_sample(logits, p=0.9):
    """Top-p (核采样) 实现"""
    probs = softmax(logits)
    sorted_idx = np.argsort(probs)[::-1]  # 概率从高到低排序
    cumsum = np.cumsum(probs[sorted_idx])
    # 找到累积概率 ≥ p 的最小 token 集合
    n_keep = np.searchsorted(cumsum, p) + 1

    # 只保留前 n_keep 个，其余置零，重归一化
    mask = np.zeros_like(probs)
    mask[sorted_idx[:n_keep]] = 1
    probs_filtered = probs * mask
    probs_filtered /= probs_filtered.sum()

    return probs_filtered, n_keep

# 测试两种分布
logits_c = np.array([5.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])
logits_f = np.array([2.0, 1.9, 1.8, 1.7, 1.6, 1.5, 1.4, 1.3])

for name, logits in [("集中分布", logits_c), ("分散分布", logits_f)]:
    probs_filtered, n_keep = top_p_sample(logits, p=0.9)
    print(f"{name}: Top-p=0.9 保留 {n_keep} 个 token")
    print(f"  过滤后概率: {probs_filtered.round(3)}\n")


> **关键洞察**：Top-p 的美妙之处在于自适应——**p 值不变，候选集大小随上下文自动调整**。信息充分时专注（候选少），信息不足时开放（候选多）。这就是为什么 Top-p 取代 Top-k 成为 SOTA 标准：它不需要手动调 k。

🔗 **AI 连接**：第 30 章 Transformer Block 的每个 Decode 步末尾都要经过解码策略选择下一个 token。第 35 章 Agent 工具调用时，Top-p=0.9 + T=0.1~0.3 是推荐的组合配置。

---


### 33.5　惩罚机制（Repetition Penalty）—— 防止车轱辘话

在长文本生成（特别是 CoT 推理）中，模型容易反复说同样的内容。**Repetition Penalty** 在每一步对已生成的 token 做 logit 惩罚：`logits[already_generated] -= penalty`。

不同于 `no_repeat_ngram_size`（硬禁止——n-gram 出现过的 token 直接禁止），penalty 是软约束——只是降低概率，不完全禁止。这给了模型"自由裁量权"：如果某个 token 确实是最合适的，即使出现过也可以再次使用，只是门槛变高了。

📐 **定义　Repetition Penalty**：对历史中出现过的 token 的 logit 减去 penalty 值（典型 1.0~2.0）。软约束——允许重复但提高门槛。防止模型陷入"复读机"循环。

💻 **代码　实现 repetition_penalty + 效果验证**


In [ ]:
import numpy as np

def softmax(x):
    x = np.float64(x); x = x-x.max(); e = np.exp(x); return e/e.sum()

vocab_size = 10
logits = np.ones(vocab_size) * 2.0  # 扁平分布——所有 token 初始概率接近
generated = [0, 3, 0, 7]  # 已经生成了 0, 3, 0, 7

# 无惩罚
probs_no_penalty = softmax(logits)
print("无惩罚:")
print(f"  Token 0 (已出现2次) 的概率: {probs_no_penalty[0]:.1%}")
print(f"  Token 5 (从未出现) 的概率: {probs_no_penalty[5]:.1%}")

# 有惩罚：对每个出现过（至少一次）的 token 统一降权一次
penalty = 1.5
logits_penalized = logits.copy()
for token_id in set(generated):
    logits_penalized[token_id] -= penalty

probs_penalized = softmax(logits_penalized)
print(f"\n有惩罚 (penalty={penalty}):")
print(f"  Token 0 (已出现2次) 的概率: {probs_penalized[0]:.1%}")
print(f"  Token 5 (从未出现) 的概率: {probs_penalized[5]:.1%}")
print(f"\n→ 已生成 token 被抑制，模型被迫'想点新的'")

> **关键洞察**：Repetition Penalty 是防止 Agent 在长思考中"车轱辘话来回说"的最后防线。在 CoT（Chain of Thought）推理中，模型可能反复回到同一个论点——penalty 强制它继续推进推理链条。

---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. 对同一个 Prompt，用四种策略各生成 100 段文本，计算 Distinct-4 指标并排序（预期：Top-p > Temperature > Top-k > Greedy）。
2. 实现 `repetition_penalty` 的代码，并在 `"I love"` 开头下验证重复被抑制。
3. 画出一张 `Temperature × Distinct-4` 的曲线图，找到"多样性-连贯性"的帕累托最优区间。
---
> 🔗 **章末钩子**：我们终于知道怎么让模型生成更像"人话"了，但这些能力是预训练给的。如果我们要让模型学会**新技能**（比如用特定格式调用工具），该怎么"教"它？→ 引向第 34 章。
> 预览：下一章——**微调落地的"手术刀"**，从 PyTorch 基础到 LoRA。
